### Introduzione al Web Caching
Il **web caching** è la tecnica utilizzata per salvare copie di risorse web (come pagine HTML, immagini, video) su server intermedi detti cache server, di modo da servire più velocemente le richieste degli utenti e ridurre il carico sui server originali.

Il problema principale che il web caching cerca di risolvere è il **data delivery bottleneck**: quando molti utenti richiedono le stesse risorse, questo può portare a latenza, congestione di banda e sovraccarico del server originale. 

Il meccanismo base è il seguente: 
- l'utente richiede una risorsa web
- la cache controlla se ne possiede una copia aggiornata, se sì la restituisce subito (**cache hit**), altrimenti la scarica dal server originale, la memorizza e la restituisce all'utente (**cache miss**).

In questo modo, se in futuro un altro utente dovesse richiedere la stessa risorsa al cache server, questa potrà essere servita immediatamente.

I cache server seguono inoltre una gerarchia: cache nel browser (locale nel dispositivo dell'utente), cache proxy dell'ISP, e infine le **CDN (Content Delivery Network)** che sono vere e proprie reti di cache distribuite geograficamente (come quelle di Akamai o Cloudflare).

Guardando il tutto da un punto di vista distribuito, possiamo vedere il **Cache System** come un insieme di **nodi cache** server che devono essere disposti nella rete in modo che ogni **nodo user** abbia qualche nodo cache vicino a cui rivolgersi per le richieste. Si vuole organizzare la rete in modo che questa gestisca nel modo più efficiente possibile le richieste.

<img src="img/cache_system.png" alt="Cache System" width="300">

"Gestire nel modo più efficiente possibile" significa potenzialmente cercare di affrontare molti problemi:
- gestire correttamente un sistema enorme e distribuito, senza un controllo centrale unico
- nodi diversi possono avere informazioni incoerenti tra loro 
- il sistema deve scalare bene all'aumentare di utenti e server
- bisogna evitare hotspot (server molto richiesti) sovraccarichi (fenomeno di "swamping")
- si deve ridurre il traffico della rete, mantenendo basso il tempo necessario per rispondere alle query degli utenti
- si vuole garantire load balancing tra i nodi cache

Il punto importante è che **Internet è un sistema dinamico**, quindi molti server che si spengono o che si aggiungono, utenti che vedono insiemi diversi di server, traffico non uniforme etc... come progettare un sistema di caching che funzioni bene in questo contesto? Serve un modo rapido, deterministico, condiviso da tutti per decidere quale cache è responsabile di ogni item --> la soluzione è usare **funzioni di hashing**.

### Hashing per il Web Caching
Nel modello CSM Cache System Model introduciamo tre insiemi:
- $I$: **insieme di item** (risorse web)
- $U$: **insieme di utenti** (client che fanno richieste)
- $S$: **insieme di server cache** (nodi che memorizzano le risorse)

In CSM si fa la forte assunzione che **gli itam siano richiesti tutti con la stessa probabilità**. L'obiettivo è distribuire uniformemente il carico delle richieste tra i server cache --> la soluzione è definire una funzione hash $h: I \rightarrow S$ che mappa ogni item a un server cache.

Se si ha $f(i) = b$, allora diremo che il server $b$ è responsabile dell'item $i$. Il funzionamento operativo è il seguente:
- quando l'utente vuole richiedere un item $i$, calcola $h(i)$ per determinare quale server cache è responsabile di quell'item
- L'utente quindi contatta $h(i)$, se il server ha già una copia dell'item la restituisce subito, altrimenti lo scarica dal server originale, lo memorizza e lo restituisce all'utente. Le richieste successive per lo stesso item saranno gestite direttamente dalla cache, senza coinvolgere di nuovo il server originale.

Il punto concettuale fondamentale perché questo meccanismo funzioni è che **la funzione hash sia condivisa tra tutti gli utenti e i server**.

Usare una funzione hash è la scelta più naturale perché ogni item (che è una stringa URL) viene convertito tramite hashing in una stringa di lunghezza fissa (usando ad esempio SHA-1, MD5, etc..) che può essere interpretata come un numero. Se si ha $n$ server cache, si può prendere il risultato del hash modulo $n$ per ottenere un indice che identifica il server responsabile di quell'item.
$$\text{server}(i) = h(i) \mod n$$
Tra l'altro utilizzando una funzione hash crittografica si ottiene una distribuzione uniforme degli item tra i server, evitando che alcuni server siano sovraccarichi mentre altri sono poco utilizzati.

Il problema di questo schema è che questo dipende direttamente dal numero di server $n$: **per la natura dinamica di Internet, se un server cache viene aggiunto oppure fallisce allora $n$ cambia!** Questo comporta anche un **cambiamento drastico della funzione di assegnazione e quindi della mappatura degli item ai server**.

L'immagine qui sotto di reshuffling mostra proprio questo fenomeno: immaginando di utilizzare una funzione hash che assegna un item all'hash dell'item + 1 modulo $n$, quando si passa da $n = 4$ a $n = 5$ server, la funzione hash passa da
$$h(d) = d + 1 \mod 4$$
a
$$h(d) = d + 1 \mod 5$$
e quasi tutti i documenti vengono assegnati a server diversi (punti neri mod 4, quadrati bianchi mod 5 --> solo un punto resta assegnato allo stesso server dopo l'aggiunta del nuovo server!):

<img src="img/reshuffling.png" alt="Reshuffling" width="400">

Tutto ciò è un problema enorme perché **in Internet non è realistico assumere che tutti gli utenti aggiornino immediatamente la loro funzione hash** --> alcuni utenti potrebbero avere ancora la vecchia hash, nel momento della ricerca di un item potrebbero contattare un server cache che non la possiede --> **cache miss** e quindi dover contattare il server originale, aumentando latenza e traffico. Questo fenomeno ripetuto per molti utenti porta quindi a cache miss storm, origin server overload etc...

### Consistent Hashing
Possiamo in modo non proprio preciso ma intuitivo definire il problema per cui un nodo con hash non aggiornato ha una visione inconsistente della rete come problema di **view inconsistency**. La soluzione per ridurre questo problema è il **consistent hashing**.

L'idea rivoluzionaria è la seguente: **sia i server che gli item vengono mappati (hashati) nello stesso spazio geometrico, un cerchio unitario $[0,1)$**. Dopo aver posizionato tutto sul cerchio, un item viene assegnato **al primo server incontrato muovendosi nel cerchio in senso orario**.

L'immagine mostra questo meccanismo: i pallini vuoti sono gli item, quelli pieni i server, e un documento viene inserito nel primo server successivo in senso orario. 

<img src="img/consistent.png" alt="Consistent Hashing" width="350">

La **differenza cruciale** rispetto ad hash(key) mod n è che **aggiungere un nuovo server non cambia completamente la mappatura degli item ai vari server, ma solo una piccola frazione di item (in particolare, quelli che cadono nell'arco immediatamente precedente al nuovo server). Tutti gli altri item restano assegnati agli stessi server!**

In questo senso definiamo due proprietà importanti per il consistent hashing:
1. **Smoothness**: quella detta qua sopra; quando si aggiunge un nuovo server (o un server viene rimosso), solo una piccola frazione di item viene riassegnata a quel server, mentre la maggior parte degli item rimane assegnata agli stessi server di prima.
2. **Balance**: se la funzione hash distribuisce uniformemente i punti sul cerchio --> anche gli item saranno distribuiti abbastanza uniformemente tra i server, evitando sovraccarichi

Introduciamo ora, informalmente, due concetti particolarmente importanti nei sistemi di caching distribuiti. Sappiamo come detto che nel mondo distribuito è possibile che in un dato momento client diversi possano vedere un insieme di server cache diversi (hanno diverse **views**). Definiamo informalmente due concetti, che poi riprenderemo meglio successivamente:
- **Spread**: Considerando tante possibili view diverse, uno stesso item non dovrebbe essere mandato a tantissimi server diversi, ma solo a un piccolo insieme di server candidati. Intutitivamente consistent hashing permette di tenere basso lo spread perché ad es. se un utente non aggiornato ha view $V_1 = \{A, B, C, D\}$ mentre un utente aggiornato $V_2 = \{A, B, C, D, E\}$, allora assumendo che $E$ sia il nuovo server aggiunto e il server subito dopo in senso orario sia $A$, allora gli unici server per cui $V_1$ e $V_2$ differiscono sono $A$ e $E$ (perché alcuni degli item che stavano su $A$ ora stanno su $E$).
- **Load**: a differenza dello spread riguarda il punto di vista del server, non dell'item. Il load chiede in parole povere: "considerato un certo server e tutte le view possibili in un dato istante, di quanti item può quel server diventare responsabile?" --> se il load è alto, significa che quel server potrebbe diventare un hotspot in alcune view, e quindi essere sovraccarico.

### Consistent Hashing: Formalizzazione
Definiamo anzitutto gli oggetti del modello:
- $C = [0,1)$: rappresenta lo spazio dei punti nel cerco unitario, dove saranno mappati sia i server che gli item
- $B$ = insieme dei server cache, con $|B| = l$
- $I$ = insieme degli item, con $|I| = n$
- Una **view** $V$ è un sottinsieme dei server, cioè l'insieme dei server che un certo utente vede come attivi in un dato momento: $V \subseteq B$. In generale considereremo sempre una certa collezione di view $\mathcal{V} = \{V_1, V_2, ..., V_k\}$ che rappresenta le diverse conoscenze dei client in un dato istante.

Secondo queste definizioni quindi, **la funzione hash dipende anche dalla view**, si ha:
$$f: 2^B \times I \rightarrow B$$
ossia ogni funzione hash mappa un certo item $i \in I$ a un server $b \in B$, ma in base a una delle possibili $2^l$ view che quel client potrebbe avere. Si scrive $2^B$ perché una view è un sottinsieme di $B$ e con $2^B$ indichiamo l'insieme delle parti (tutti i possibili sottinsiemi) di $B$.

Per comodità, invece di scrivere $f(V, i)$, scriveremo semplicemente $f_V(i)$ per indicare che la funzione hash dipende dalla view $V$.

Formalizziamo ora le quattro proprietà fondamentali che vorremmo che la funzione hash $f$ soddisfi:
1. **Monotonicity**: si tratta della proprietà più importante contro il reshuffling. Informalmente ci dice che, se $V \subseteq W$ (ossia $W$ contiene tutti i server di $V$ più eventualmente server nuovi) allora gli item hanno solo due possibilità: o restano assegnati allo stesso server, oppure vengono assegnati a un nuovo server che è in $W$ ma non in $V$, cioè **non possono essere "reshuffled" verso vecchi server diversi**. Formalmente:
    $$f_W(i) \in V \Rightarrow f_V(i) = f_W(i)$$
    (cioè se nella view nuova $W$ l'item è ancora assegnato a un vecchio server $V$, allora doveva essere esattamente nello stesso server anche nella view vecchia $V$)
2. **Balance**: gli item devono essere distribuiti quasi uniformemente tra i server, per evitare sovraccarichi. Formalmente:
    $$\Pr_f(f_V(i) = b) = \widetilde O\left(\frac1{|V|}\right)$$
    (dove $\widetilde O$ è detto O-tilde o soft-O e indica che stiamo usando notazione asintotica ignorando fattori polilogaritmici, quindi in parole povere stiamo dicendo che ogni server ha probabilità approssimativamente uniforme $1/|V|$ di essere ricevere un item. La $f$ sotto la probabilità fa riferimento al fatto che stiamo considerando una famiglia di funzioni hash, e la probabilità è presa su una scelta casuale di una funzione hash da quella famiglia, vedi meglio sotto per ripasso)
3. **Spread**: per un item fissato $i$, si guarda **quante assegnazioni diverse può ricevere quell'item nelle varie view**. Intuitivamente avere spread alto è molto problematico per diversi motivi: se un item è assegnato a server diversi in diverse view allora è necessario creare tante copie dello stesso item (una per ogni server a cui è assegnato) aumentando lo spazio occupato nella cache, e in particolare all'inizio quando un item è assegnato solo a un server allora i cache miss per quell'item da parte dei nodi non aggiornati sono molto frequenti etc... **avere spread piccolo quindi intuitivamente significa che anche se gli utenti non sono perfettamente sincronizzati ed hanno view diverse, uno stesso item sarà comunque associato solo a pochi server possibili**. 
Formalmente:
    $$\text{Spread}(i) = |\{f_{V_j}(i)\}_{j=1}^k|$$
    ossia quatni server distinti compaiono assegnando $i$ nelle varie view.
1. **Load**: per un server $b$ fissato, si guarda **quanti item distinti possono essere assegnati a $b$ in almeno una delle view**. 
   $$Load(b) = |\{i \in I : \exists V \in \mathcal{V} \text{ tale che } f_V(i) = b\}|$$

**Interpretazione concreta dello spread**: lo spread è poi in pratica il numero di copie che che il server centrale dovrà distribuire di uno stesso item sulle varie cache. Infatti se utenti diversi hanno view diverse, lo stesso item potrebbe essere assegnato a server diversi e quindi il server centrale deve fornire copie dell'item a tutti quei server. Spread grande significa in questo senso molto traffico e memoria allocata (**swamping**)

Riguardo la **monotonicity**, come visto prima, l'uso di hashing classico con modulo $n$ comporta un reshuffling pressoché totale degli item anche tra server della stessa vista prima dell'aggiornamento. Al contrario il consistent hashing soddisfa monotonicity perché quando si aggiunge un nuovo server, gli unici item che cambiano assegnazione sono quelli che erano assegnati al server immediatamente precedente in senso orario, che ora vengono assegnati al nuovo server. Tutti gli altri item restano assegnati agli stessi server di prima, quindi non c'è reshuffling verso altri server diversi da quello nuovo.

Riguardo **balance**, questa è garantita se si sceglie una buona famiglia di funzioni hash universali (si parla di famiglia di funzioni hash perché l'analisi è probabilistica, quindi il sistema ne sceglie una a caso dalla famiglia; universale fa invece riferimento al fatto che la famiglia è progettata per evitare troppe collisioni. In particolare ricordiamo che una famiglia hash $\mathcal{H} = \{h_1, h_2, ..., h_m\}$ è universale se per ogni coppia di chiavi distinte $x, y$ si ha: $\Pr_{h \in \mathcal{H}}(h(x) = h(y)) \leq \frac{1}{d}$, dove $d$ è il numero di bucket). 

Nota che una famiglia hash universale garantisce bene la balance, ma non garantisce automaticamente anche monotonicity. Infatti una funzione tipo $f_{a,b} = ax + b \mod p$ (dove $p$ è un numero primo) è una funzione hash universale che quindi distribuisce bene gli item, ma se cambia il numero di server/bucket $p$ allora può comunque causare un grande reshuffling.

Si dimostra ora un Lemma che formalizza matematicamente l'idea intuitiva di **stabilità** del consistent hashing, quella per cui se cambia la view dei server il numero di item che restano assegnati allo stesso server è alto in media (e quindi una frazione significative degli item rimane stabile). 

> **Lemma 1**
> 
> Sia $\mathcal{F}$ una famiglia di funzioni hash monotonica e bilanciata. Siano $V, W \in \mathcal{V}$ due views. Allora si ha che
> $$\mathbb{E}_{f \in \mathcal{F}}\left[\frac{|\{i \in I : f_V(i) = f_W(i)\}|}{n}\right] = \Omega\left(\frac{|V\cap W|}{|V \cup W|}\right)$$
> ossia il valore atteso della frazione di item che rimane stabile (assegnata allo stesso server, frazione perché al denominatore abbiamo $|I|=n$) è la Jaccard similarity tra le due view $V$ e $W$ (e quindi proporzionale alla sovrapposizione, somiglianza tra le due view). Infatti intuitivamente se due view sono quasi uguali (condividono quasi tutti i server) --> quasi tutti gli item restano stabili
>
> **Dimostrazione**
>
> L'idea generale è, invece di vedere quanti item restano stabili passando direttamente da $V$ a $W$, fare un passaggio intermedio:
> $$V \rightarrow V \cup W \rightarrow W$$
> cioè prima aggiungiamo a $V$ i server che sono in $W$ ma non in $V$, e poi rimuoviamo quelli che erano in $V$ ma non sono in $W$.
>
> Quando passiamo da $V$ a $V \cup W$, per **monotonicity**, gli item che possono cambiare assegnazione sono solo quelli che, nella view più grande $V \cup W$, finiscono in un server che non era in $V$, quindi di $W \setminus V$. Grazie alla **balance**, ogni server riceve circa la stessa frazione di item, e poiché nella view $V \cup W$ ci sono $|V \cup W|$ server, la frazione attesa di item assegnati ai nuovi server è circa $O\left(\frac{|W \setminus V|}{|V \cup W|}\right)$. Quindi la frazione di item che cambia in questo passaggio è al più 
> $$O\left(\frac{|W \setminus V|}{|V \cup W|}\right)$$
>
> Quando passiamo da $V \cup W$ a $W$ stiamo rimuovendo i server presenti in $V$ ma non in $W$: $V \setminus W$. Se un item era assegnato a un server di $W$ allora non ha motivo di essere riallocato. Al contrario gli item che dovranno cambiare sono quelli che erano assegnati a server di $V \setminus W$, che poi vengono rimossi. Grazie alla balance, la frazione di item assegnati a quei server è circa $O\left(\frac{|V \setminus W|}{|V \cup W|}\right)$.
>
>Sommando quindi, il numero di item che in tutto può cambiare passando da $V$ a $W$ è al più:
>$$O\left(\frac{|W \setminus V|}{|V \cup W|} + \frac{|V \setminus W|}{|V \cup W|}\right)$$
>si osserva che $|W \setminus V| + |V \setminus W|$ è esattamente la parte non comune tra $V$ e $W$, mentre $|V \cap W|$ è proprio la parte comune. Quindi più grande è la parte comune, più piccola è la parte non comune, e quindi più item restano stabili. Per questo si scrive che la frazione stabile è:
>$$ 1- O\left(\frac{|W \setminus V| + |V \setminus W|}{|V \cup W|}\right) = \Omega\left(\frac{|V\cap W|}{|V \cup W|}\right)$$
>
> $\blacksquare$

### **DVH Hypothesis** e ripetizioni di roba importante
**Density View Hypotesis DVH**: si assumerà che **per ogni view $V$ si avrà che:**
$$|V| \geq \frac{|B|}{t}$$
ossia ogni client vede almeno una frazione significativa dei server totali, dove $t \geq 1$ è un parametro. Se $t=1$ allora ogni view vede praticamente tutti i server, mentre al crescere di $t$ le view possono essere più "incomplete". 

**Questa è un'ipotesi fondamentale e assolutamente necessaria in quanto se le view fossero troppo piccole o del tutto arbitrarie (avversariali) --> sarebbe impossibile controllare spread e load.**

In realtà, anche se ogni view ha dimensione almeno $|B|/t$, è comunque possibile costruire $t$ view disgiunte $V_1, V_2, ..., V_t$ che non condividono nessun server tra loro. In questo caso lo stesso item potrebbe essere assegnato a un server diverso per ogni view, e quindi $Spread(i) = t$, ossia cresce linearmente nel numero di view. Quindi anche con l'ipotesi il problema non è assolutamente banale, per evitare questa disgrazia è necessario **progettare una famiglia hash che limiti questa crescita** (e vedremo che consistent hashing è proprio una di queste famiglie).

In particolare abbiamo come obiettivo per Consistent Hashing ottenere, date $k$ view totali possibili in un dato istante, che lo spread cresca solo logaritmicamente nel numero di view, e quindi molto lentamente --> enorme miglioramento di scalabilità.

Approfittiamo per ricordare la definizione di **Load**: $Load_{\mathcal{F}}(\mathcal{V}, b)$ è il **numero totale di item che possono finire nel server $b$ considerando tutte le possibili view**. 

Anche se non evidentissimo **Spread** e **Load** sono concetti piuttosto diversi: mentre lo spread si focalizza su un item e chiede "quanti server diversi possono ricevere questo item nelle varie view?", il load si focalizza su un server e chiede "quanti item diversi possono essere assegnati a questo server in almeno una delle view?". Il controesempio estremo che ne mostra la differenza è il seguente: immaginiamo che tutti gli item finiscano sempre nello stesso server $A$ in tutte le view. Allora sicuramente $Spread(i) = 1$ per ogni $i\in I$, ma il server $A$ riceve tutti gli item, quindi $Load(A) = |I| = n$, che è il massimo possibile. Quindi avere spread basso non implica necessariamente anche load basso, e viceversa.

In generale il nostro obiettivo è quindi costruire una **famiglia di consistent hashing** che abbia:
- **good balance**: distribuisce bene gli item tra i server
- **monotonicity**: evita reshuffling massivo quando cambia la view
- **low spread**: limita la dispersione degli item tra server diversi nelle varie view
- **low load**: evita il sovracccarico di alcuni server 

### (Informal) Consistent Hashing Family
La famiglia concreta di consistent hashing che verrà studiata è la famiglia $\mathcal{UC}_{rnd}$ (**Unit Circle Randomized Hash Family**). 

L'idea è la seguente:
- Si scelgono **due funzioni random** che mappano rispettivamente **gli item $i \in I$ e i server $b \in B$ in punti casuali del cerchio unitario** $C = [0,1)$:
$$r_I: I \rightarrow C \qquad r_B: B \rightarrow C$$
- L'assegnazione degli item ai server avviene come segue: **un item $i$ viene assegnato al primo server incontrato muovendosi nel cerchio in senso orario, a partire dal punto $r_I(i)$**.

Quindi formalmente la **famiglia Unit Circle Randomizeed Hash Family** è nell'insieme di tutte le possibili scelte di $r_I$ e $r_B$, ossia:
$$\mathcal{UC}_{rnd} = \{(r_I, r_B) \quad \text{ tali che } \quad r_I: I \rightarrow C, r_B: B \rightarrow C\}$$
Una volta scelta casualmente una coppia $(r_I, r_B)$, questa definisce una specifica funzione di assegnazione degli item ai server, e userò da quel momento in poi sempre quella coppia per tutte le view, in modo che tutti i client condividano la stessa funzione hash. Il punto di usare una famiglia di funzioni hash è che nel modello teorico **scegliendo randomicamente una coppia $(r_I, r_B)$ da $\mathcal{UC}_{rnd}$, otteniamo una funzione hash che mappa in modo perfettamente uniforme e indipendente gli item e i server su punti casuali del cerchio**.

**La randomness in particolare è fondamentale per proteggere da comportamenti avversariali**: supponiamo infatti che un avversario possa far crashare alcuni server. Se la disposizione dei server nel cerchio fosse nota e deterministica, fissa a priori, allora l'avversario (che conosce il codice, come tutti) potrebbe ad esempio eliminare tutti i server in una certa zona del cerchio, creando un grosso gap e costringendo quindi tutti gli item a finire sul primo server dopo il gap e a sovraccaricarlo. 

Se invece la disposizione dei server è casuale, l'avversario conosce il codice ma non sa dove saranno mappati i server nel momento dell'esecuzione --> non può attaccare efficacemente il sistema.

Qui di seguito si discute solo intuitivamente perché $\mathcal{UC}_{rnd}$ (grazie anche alla geometria del cerchio) rendono possible ottenre le proprietà desiderate:
- **Balance**: dal momento che gli item e i server sono distribuiti uniformemente --> ogni server controlla circa la stessa lunghezza di arco nel cerchio --> il carico tende a distribuirsi uniformemente tra i server
- **Monotonicity**: questo deriva in modo del tutto naturale grazie alla geometria del cerchio e alla regola di assegnazione: quando si aggiunge un nuovo server $b$, cambiano solo gli item che erano assegnati al server immediatamente precedente a $b$ in senso orario, che ora vengono assegnati a $b$. Tutti gli altri item restano assegnati agli stessi server di prima, quindi non c'è reshuffling verso altri server diversi da quello nuovo (gli altri item restano stabili!).
- **Load**: un server riceve principalmente gli item vicini a lui sul cerchio, quindi anche con molte view diverse il suo load non esplode.
- **Spread**: lo spread resta controllato per lo stesso motivo; dato che un item può essere potenzialmente assegnato solo a pochi server vicini sul cerchio, il numero di server diversi a cui può essere assegnato quell'item nelle varie cresce lentamente.

In particolare con hashing tradizionale, **spread e load possono crescere anche linearmente nel numero $k$ di view attive in un dato momento**, mentre dimostreremo come con $\mathcal{UC}_{rnd}$ si riesce a ottenere Spread e Load **con crescita logaritmica rispetto a $k$** (**SEMPRE SOTTO DVH**).

#### Breve approfondimento sulla questione randomness
L'idea teorica è quindi questa: abbiamo a disposizione la famiglia $\mathcal{UC}_{rnd}$ che contiene tutte le possibili coppie di funzioni random $r_I$ e $r_B$ che mappano gli item e i server su punti del cerchio, e una volta scelta una coppia in modo u.a.r. dalla famiglia questa mapperà sempre allo stesso modo gli item e i server, di modo che la coppia possa essere condivisa tra tutti i client e che quindi il sistema possa funzionare correttamente.

Ma in pratica come si implementa una cosa del genere?  
Sappiamo bene da Big Data che se volessimo una funzione hash che distribuisca davvero gli elementi nei bucket in modo perfettamente uniforme e indipendente sarebbe impraticabile. Se volessimo distribuire $n$ oggetti, sarebbe necessario generare randomicamente **e memorizzare** una delle $n!$ possibili permutazioni degli oggetti, di modo che questa possa essere utilizzata per mappare ogni oggetto a un punto del cerchio (es. $\pi = (3,5,1,4,2)$ significa che l'oggetto 1 viene mappato al punto 3, l'oggetto 2 al punto 5, etc...). Scegliere uniformemente una permutazione tra le $n!$ possibili richiede almeno $log(n!) = O(n \log n)$ (Stirling) bit random (perché genero ogni bit u.a.r., e il numero che rappresento è dell'ordine di $n!$), ma il problema vero è che **serve anche memorizzare la permutazione** --> spazio necessario $O(n \log n)$, intrattabile per $n$ grande.

Per questo motivo nella pratica le funzioni hash del modello teorico, che mapperebbero perfettamente gli item e i server su punti casuali del cerchio, non possono essere implementate direttamente così. Una buonissima approssimazione che si fa nella realtà è quindi utilizzare **funzioni hash crittografiche come SHA-1, MD5, etc...** che mappano ogni stringa (URL o nome del server) a una stringa di lunghezza fissa (es. 160 bit per SHA-1), che può essere interpretata come un numero intero che viene poi normalizzato nell'intervallo $[0,1)$. In questo senso però dove sta la famiglia? **Nei seed delle funzioni hash crittografiche!** La famiglia rappresenta in questo senso, concettualmente, l'insieme di tutte le possibili hash function che potrei ottenere cambiando seed. Quindi scegliendo il seed casualmente, ottengo una funzione hash che mappa in modo pseudo-casuale gli item e i server su punti del cerchio, ma lo farà sempre nello stesso modo, quindi sarà condivisibile tra tutti i client.

Ricorda poi ovviamente che i punti nel cerchio sono infiniti, quindi deve esistere tutta una procedura di discretizzazione affinché i server e gli item vengano mappati a un numero finito di punti, ma questo è un dettaglio tecnico che non cambia la sostanza del discorso.

### **Rigorous Analysis of the $\mathcal{UC}_{rnd}$ Family**
Ricordiamo al volo il setup: $C = [0,1)$, $B$ = insieme dei server cache, con $|B| = l$, $I$ = insieme degli item, con $|I| = n$, e una collezione di view $\mathcal{V} = \{V_1, V_2, ..., V_k\}$ che rappresenta le diverse conoscenze dei client in un dato istante.

Introduciamo una novità nella definizione formale della costruzione: **le copie dei server**. Abbiamo finora considerato per semplicità che ogni server fosse mappato biunivocamente a un punto del cerchio, quando nelle applicazioni reali **ogni server $b$ possiede $m$ copie virtuali** (**virtual nodes**) mappate anch'esse su punti casuali del cerchio. Si modifica quindi la definizione di $r_B$:
$$r_B: B \times [m] \rightarrow C$$
dove $[m] = \{1, 2, ..., m\}$ rappresenta le $m$ copie virtuali di ogni server. L'idea è che ogni server non è rappresentato da un solo punto sul cerchio, ma da $m$ punti diversi, e quando un item viene assegnato a un server, viene assegnato alla copia virtuale più vicina in senso orario.

L'idea intuitiva dietro il concetto dei virtual nodes è che se ogni server avesse un solo punto casuale mappato nel cerchio, allora potrebbe capitare che alcuni server siano mappati in zone del cerchio molto piccole, e quindi ricevano pochissimi item, mentre altri server potrebbero essere mappati in zone più grandi e ricevere molti item, creando squilibri. Al contrario con $m$ copie virtuali per server il carico si uniforma meglio riducendo la varianza del numero di item assegnati a ciascun server.

Gli item chiaramente continuano ad essere mappati sempre allo stesso modo, con $r_I: I \rightarrow C$.

Formalmente come detto in precedenza, la famiglia $\mathcal{UC}_{rnd}$ è l'insieme di tutte le possibili coppie $(r_B, r_I)$, che mappano copie dei server e gli item sul cerchio:
$$\mathcal{UC}_{rnd} = \{(r_B, r_I) \quad \text{ tali che } \quad r_B: B \times [m] \rightarrow C, r_I: I \rightarrow C\}$$
**Scegliere una coppia casuale da questa famiglia è equivalente a distribuire uniformemente e indipendentemente tutti gli item e tutte le copie dei server sul cerchio.**

Per definire formalmente come avviene l'assegnazione degli item ai server considerando anche le copie virtuali, definiamo data una **view $V$** l'insieme di tutti i punti del cerchio corrispondenti alle copie dei server attivi in quella view:
$$P_V = \bigcup_{b \in V} \{r_B(b, 1), r_B(b, 2), ..., r_B(b, m)\}$$

L'assegnazione di un item funziona come segue: dato un item $i \in I$: considero il punto $r_I(i)$ sul cerchio, e mi muovo in senso orario fino a incontrare il primo punto di $P_V$, e assegno $i$ al server associato a quel punto. Formalmente:
$$f_V(i) = \text{server associato con il punto } \arg\min_{p \in P_V} \{p - r_I(i) \mod 1\}$$
dove con $p - r_I(i) \mod 1$ indichiamo la distanza in senso orario tra il punto $r_I(i)$ e il punto $p$ nel derchio, e si sceglie il punto che è più vicino in senso orario.

Il modulo 1 si usa perché il cerchio unitario è identificato dall'intervallo $[0,1)$, quindi se $r_I(i)$ è ad esempio 0.9 e $p$ è 0.1, la distanza in senso orario è 0.2 (non -0.8), e quindi si calcola come $p - r_I(i) \mod 1$.

#### Monotonicity
Si dimostra formalmente che la famiglia $\mathcal{UC}_{rnd}$ è **strettamente monotona**

> **Teorema 1 (Monotonicity)**
> 
> Sia $(r_B, r_I) \in \mathcal{UC}_{rnd}$ una qualunque funzione hash della famiglia. Siano $V \subseteq W$ due view, allora per ogni item $i \in I$ si ha che:
> $$\text{se } f_W(i) \in V \Rightarrow f_V(i) = f_W(i)$$
> ossia se si aggiunge un server (o più) alla view $V$ ottenendo la view $W$, e l'item continua ad essere assegnato a un server che era già presente nella view precedente $V$, allora quell'item è assegnato allo stesso server in entrambe le view. Al massimo quindi un item può essere riassegnato solo a server nuovi, ma non a server vecchi diversi da prima.
>
> **Dimostrazione**
>
> Siano $V \subseteq W$ due view, con $V$ la view più piccola e $W$ quella più grande. Per definizione, a ogni view $V$ è associato l'insieme dei punti dei server visibili in quella view (che includono anche le copie virtuali), e lo stesso per la view $W$:
> $$P_V = \bigcup_{b \in V} \{r_B(b, 1), ..., r_B(b, m)\} \qquad P_W = \bigcup_{b \in W} \{r_B(b, 1), ..., r_B(b, m)\}$$
> Poiché $V \subseteq W$, questo significa che tutti i server presenti nella view $V$ sono anche presenti in $W$. Ma quindi anche tutte le copie virtuali di quei server sono presenti in $P_W$ --> $P_V \subseteq P_W$.
>
> Fissiamo ora un item $i \in I$ e poniamo $r_I(i) = x$ il punto del cerchio a cui è mappato quell'item. Nella view grande $W$, l'item $i$ sarà assegnato al primo punto di server (anche potenzialmente copia) incontrato muovendosi in senso orario a partire da $x$. Chiamiamo questo punto:
> $$p^* = \arg\min_{p \in P_W} \{(p - x) \mod 1\}$$
> Quindi $p^*$ è il server (o copia del server) che diventa responsabile dell'item $i$ nella view $W$.
>
> Assumiamo ora l'ipotesi del teorema $f_W(i) \in V$, ossia il server responsabile di $i$ nella view $W$ è un server che era già presente anche nella view $V$. Quindi il punto $p^*$ apparteneva necessariamente anche a $P_V$ --> $p^* \in P_V$. Ma se $p^* \in P_V$, comunque nessun altro punto di $P_V$ può essere più vicino a $x$ di $p^*$, perché abbiamo visto prima che $P_V \subseteq P_W$, e $p^*$ è il punto più vicino a $x$ in $P_W$ (cioè in $P_V$ non è possibile che esistano altre copie di server più vicine a $x$ di $p^*$, altrimenti avrebbero dovuto essere più vicine anche in $P_W$). 
>
> Ma quindi l'item $i$ viene assegnato allo stesso server sia nella view $V$ che nella view $W$, e quindi $f_V(i) = f_W(i)$.
>
> $\blacksquare$

#### Lemma per Spread e Load: Hitting Property
Si enuncia e dimostra un lemma fondamentale per dimostrare il bound sia sullo spread che sul load.

> **Lemma 2 (Hitting Property)**
>
> Consideriamo la famiglia $\mathcal{UC}_{rnd}$ con $m \ge 1$. Sia $\mathcal{V} = \{V_1, V_2, ..., V_k\}$ una collezione di view tale che:
> $$\left|\bigcup_{j=1}^k V_j\right| = T$$
> (ossia l'insieme totale dei server distinti presenti in tutte le view è $T$).  
> Supponiamo che valga la **DVH**, per cui ogni view contiene almeno una frazione costante dei server totali:
> $$|V_j| \geq \frac{T}{t} \qquad \forall j=1, ..., k, t\ge 1$$
> Allora, preso un qualunque sottinsieme fissato (arco del cerchio unitario) $S \subseteq C$ di misura $s \ge \frac{4t\log(2Nk)}{Tm}$, il sottinsieme $S$ contiene almeno un server da ogni view $V_j$ con probabilità almeno $1 - \frac{1}{2N}$ rispetto alla scelta casuale uniforme della mappa $r_B$ dei server nel cerchio, dalla famiglia $\mathcal{UC}_{rnd}$.
>
> Intuitivamente questo Lemma ci sta dicendo che, se prendo un arco $S$ del cerchio abbastanza lungo, allora con alta probabilità ogni view avrà almeno un proprio server mappato in $S$. Nota bene che l'alea non è nella scelta di $s$, che viene fissato deterministicamente da noi, ma nella disposizione dei punti dei server sul cerchio e quindi nella scelta random della mappa $r_B$ nel momento in cui la scelgo dalla famiglia $\mathcal{UC}_{rnd}$.
>
> Interpretiamo $s \ge \frac{4t\log(2Nk)}{Tm}$. Ci sta dicendo che ovviamente all'aumentare del numero totale di server $T$ e del numero di copie virtuali $m$, allora l'arco che ci serve prendere perché la hitting property sia vera con alta probabilità diventa più piccolo! Al contrario, se le view sono "meno dense" (ogni view vede pochi dei server totali, la DVH è soddisfatta con $t$ grande) --> serve un arco più lungo per essere sicuri che ogni view abbia almeno un server in quell'arco. Allo stesso modo, all'aumentare del numero di view $k$ e del **parametro di confidenza** $N$ serve si un arco più lungo, ma con crescita logaritmica, quindi molto lenta.
>
> $N$ rappresenta come detto il parametro di confidenza: all'aumentare di $N$ aumenta la probabilità che la hitting property sia vera. La probabilità di fallimento (esistano view che non hanno server in $S$) è infatti $\frac{1}{2N}$, quindi se prendo $N = 100$ probabilità di fallimento $\le \frac{1}{200}$, se prendo $N = 1000$ probabilità di fallimento $\le \frac{1}{2000}$, etc... Quindi aumentando $N$ aumenta la probabilità che la hitting property sia vera, ma cresce anche la dimensione dell'arco, però logaritmicamente!
>
> **Typical Parameter Settings**: $N$ è una funzione lineare/poly di $n = |I|$ e/o $l = |B|$, mentre $T$ è tipicamente $T = \Theta(l)$ e $t = O(polylog(n,l))$ (dove si ricorda polylog è una funzione del tipo $O(\log^c(n,l))$)
> 
> **Dimostrazione**
>
> Fissiamo una view $V_j$. Vogliamo stimare la probabilità che nessun punto dei server di $V_j$ cada nell'arco $S$. 
>
> Sappiamo che ogni server ha esattamente $m$ copie sul cerchio, e che per DVH $|V_j| \geq \frac{T}{t}$, quindi ogni view $V_j$ ha almeno $M=\frac{Tm}{t}$ punti server mappati sul cerchio. Definiamo ora la variabile aleatoria:
> $$X_j = \text{numero di punti server di } V_j \text{ che cadono in } S$$
> Dal momento che i punti sono scelti in odo u.a.r. e indipendente sul cerchio, ogni punto cade proprio nell'arco $S$ con probabilità pari a $s$ (infatti stiamo parlando di un cerco con punti in $[0,1)$, quindi la probabilità che un punto cada in un certo arco è proprio la misura di quell'arco). 
>
> Ma quindi $X_j$ è la somma di $M$ variabili Bernoulliane indipendenti (lancio dei server sul cerchio) --> ha distribuzione binomiale $X_j \sim Bin(M, s)$, con valore atteso $\mathbb{E}[X_j] = Ms$.
>
> L'evento fallimento è quello per cui per la view $V_j$ nessun punto server cade in $S$, ossia $X_j = 0$. Dato che la media dei punti è $Ms$, dire $X_j = 0$ significa che $X_j$ è molto più piccolo della media, ha quindi senso usare una disuguaglianza di concentrazione per stimare questa probabilità. In particolare useremo un Chernoff bound per la coda di sinistra (possiamo applicarlo perché $X_j$ è una variabile binomiale, quindi somma di Bernoulli indipendenti).
>
> Anche se $X_j$ è in realtà una variabile binomiale discreta e non una gaussiana, la figura sotto aiuta a visualizzare intuitivamente l'idea alla base del Chernoff lower tail: il valore $X_j = 0$ si trova molto lontano dalla media $Ms$ e quindi la sua probabilità decade esponenzialmente.
>
> <img src="img/gauss.jpeg" alt="gauss" width="300"/>
>
> Chernoff lower tail dice che:
> $$\Pr(X_j \leq (1-\delta)Ms) \leq e^{-\frac{\delta^2 Ms}{2}} \qquad \text{per ogni } 0 < \delta < 1$$
> Per ottenere $X_j = 0$, poniamo $0 = (1-\delta)Ms$, da cui $\delta = 1$. Sostituendo otteniamo:
> $$\Pr(X_j = 0) \leq e^{-\frac{Ms}{2}} \leq 2e^{-\frac{Ms}{4}}$$
> dove l'ultima uguaglianza è vera per $Ms$ sufficientemente grande.
>
> Vogliamo che questa probabilità sia sufficientemente piccola, più precisamente:
>$$
>\begin{aligned}
>2e^{-\frac{Tm}{4t}s} \leq \frac{1}{2Nk} \iff
>e^{-\frac{Tm}{4t}s} \leq \frac{1}{4Nk} \iff
>-\frac{Tm}{4t}s \leq \log\left(\frac{1}{4Nk}\right) \\
>-\frac{Tm}{4t}s \leq -\log(4Nk) \iff
>\frac{Tm}{4t}s \geq \log(4Nk) \\
>\iff s \geq \frac{4t\log(4Nk)}{Tm} \approx \frac{4t\log(2Nk)}{Tm}
>\end{aligned}
>$$
>
> Quindi, prendendo $s \geq \frac{4t\log(2Nk)}{Tm}$, otteniamo che la probabilità che per una view $V_j$ nessun punto server cada in $S$ è al più $\frac{1}{2Nk}$.  
> Però questo non ci basta perché nel Lemma stiamo dicendo che **per nessuna delle view $V_j$ deve accadere questo evento** (tutte le view devono avere almeno un server in $S$) --> sfruttiamo Union Bound:
> $$\Pr(\text{esiste } j \text{ tale che } X_j = 0) \leq \sum_{j=1}^k \Pr(X_j = 0) \leq k \cdot \frac{1}{2Nk} = \frac{1}{2N}$$
> Quindi con probabilità almeno $1 - \frac{1}{2N}$, ogni view $V_j$ ha almeno un server in $S$.
>
> $\blacksquare$

#### Bound sullo Spread
Si ricorda che lo spread di un item $i$ è:
$$Spread_f(\mathcal{V}, i) = |\{f_V(i) : V \in \mathcal{V}\}|$$
ossia guardo tutte le view $V_1, \ldots, V_k$, assegno l'item $i$ a ciascuna view e conto quanti server distinti possono ricevere quell'item.

> **Teorema 2 (Spread)**
> 
> Sotto DVH con parametro $t$, ogni item $i \in I$ ha $Spread_{f \in \mathcal{UC}_{rnd}}(\mathcal{V}, i) = O(t \log(Nk))$ con probabilità almeno $1 - \frac{1}{N}$ rispetto alla scelta random di $f \in_u \mathcal{UC}_{rnd}$.
>
>Stiamo cioè dicendo che lo Spread non cresce linearmente nel numero di view $k$, ma solo logaritmicamente (insieme al valore di confidenza $N$). Inoltre come al solito il fattore $t$ rappresenta la "densità" delle view: è chiaro quindi che se non si garantisce che ogni view veda abbastanza server (quindi $t$ grande) --> lo spread può essere più alto, ma se ogni view vede una buona frazione dei server totali (quindi $t$ piccolo) --> lo spread è più basso.
>
>**Dimostrazione**
>
>Dal momento che sarebbe difficile contare direttamente il numero di server diversi a cui può essere assegnato l'item $i$ per arrivare al worst case, seguiamo la seguente strategia logica:
>1. Definiamo un evento cattivo $A = \text{"il teorema è falso"}$ e due eventi cattivi $B_1$ e $B_2$, più semplici da analizzare;
>2. Si dimostra che $A \Rightarrow B_1 \cup B_2$ sfruttando de Morgan, dimostrando che $\neg B_1 \cap \neg B_2 \Rightarrow \neg A$ (se i due eventi semplici non accadono, allora di sicuro il teorema è vero);
>3. Sapendo che $A \Rightarrow B_1 \cup B_2$, ossia che se il teorema è falso allora deve essere successo almeno uno tra $B_1$ e $B_2$, dimostriamo che $B_1$ e $B_2$ accadono con probabilità piccola e che quindi il teorema è vero con alta probabilità.
>
> Intuitivamente infatti: dire $A \Rightarrow B_1 \cup B_2$ significa dire che se $A$ avviene allora almeno uno tra $B_1$ e $B_2$ deve avvenire --> se $B_1$ e $B_2$ non avvengono allora di sicuro neanche $A$ può avvenire.
>
> <img src="img/venn.jpeg" alt="venn" width="300"/>
>
> Ragionando in termini probabilistici nello spazio di probabilità: se la probabilità che $B_1$ e $B_2$ avvengano è bassa --> la probabilità che $A$ avvenga è ancora più bassa --> il Teorema è vero con alta probabilità $\left(\Pr(A) \leq \Pr(B_1 \cup B_2)\right)$.
>
> Definiamo quindi formalmente questi eventi, per ogni possibile item $i$:
> $$A = \text{il teorema è falso} = \{Spread_{f}(\mathcal{V}, i) > 8t \log(4Nk)\}$$
> $$ B_1 = \text{esiste una view } V_j \text{ senza server point dentro }S$$
> $$ B_2 = \{X > 8t \log(4Nk)\}$$
> dove per $B_2$, $X$ è il numero totale di server point dentro l'arco $S$, ed $S$ è l'arco del cerchio scelto subito a destra di $r_I(i)$ (punto in cui è stato mappato l'item $i$), con lunghezza:
> $$s = \frac{4t\log(4Nk)}{Tm}$$
>
> **Dimostriamo anzitutto che $\neg B_1 \cap \neg B_2 \Rightarrow \neg A$**:
>   - **se non accade $B_1$**, allora ogni view ha almeno un server point dentro $S$ --> in ogni view l'item $i$ sarà assegnato a un server che si trova in $S$, dal momento che abbiamo definito $S$ come l'arco immediatamente successivo a $r_I(i)$.
>   - **se non accade $B_2$**, allora in $S$ non ci sono troppi server point: $X \leq 8t \log(4Nk)$.
>
>Quindi l'item $i$, nelle varie view, può essere assegnato solo ai server presenti in $S$, e questi sono al più $X$, perciò lo Spread (numero di server diversi a cui può essere assegnato $i$ nelle varie view) è al più $X$:
>$$Spread_f(\mathcal{V}, i) \leq X \leq 8t \log(4Nk)$$
> Abbiamo quindi dimostrato che $\neg B_1 \cap \neg B_2 \Rightarrow \neg A$, e quindi che $A \Rightarrow B_1 \cup B_2$ --> se il Teorema fallisce, deve essere successo almeno uno tra $B_1$ e $B_2$. Vediamo ora perché la probabilità che $B_1$ o $B_2$ avvengano è piccola:
> 1. $B_1$ ha probabilità piccola grazie al Lemma 2 (Hitting Property): prendendo $s = \frac{4t\log(4Nk)}{Tm}$, otteniamo che con probabilità almeno $1 - \frac{1}{2N}$ ogni view ha almeno un server point in $S$ --> $B_1$ accade con probabilità al più $\frac{1}{2N}$.
> 2. $B_2 = \{X > 8t \log(4Nk)\}$, dove $X$ conta il numero di server point che cadono nell'arco $S$. Poiché i server point sono messi uniformemente a caso sul cerchio, $X$ è una variabile binomiale $X \sim Bin(Tm, s)$ ($Tm$ server totali, con probabilità $s$ di cadere in $S$).  
> La media è quindi 
>       $$\mathbb{E}[X] = Tms = Tm \cdot \frac{4t\log(4Nk)}{Tm} = 4t \log(4Nk)$$
>       Sia questa media $\mu$. Ma $B_2$ dice che $X \gt 8t \log(4Nk) = 2\mu$, ossia $B_2$ chiede che $X$ sia più grande del doppio della sua media. Ha senso quindi usare il Chernoff Bound per la coda di destra (nelle slide si usa un bound più leggero, per questo qui non usiamo subito il classico. Ricorda inoltre che possiamo applicare Chernoff perché $X$ è una variabile binomiale, quindi somma di Bernoulli indipendenti):
>       $$\Pr(X \ge 2\mu) \leq e^{-\frac{\mu}{3}} \le 2e^{-\frac{\mu}{4}}$$
>       sostituendo $\mu = 4t \log(4Nk)$ otteniamo:
>       $$\Pr(X \ge 2\mu) = \Pr(B_2) \leq 2e^{-t \log(4Nk)} \le 2e^{- \log(4Nk)}$$
>       dove l'ultima disuguaglianza vale perchè $t \ge 1$. Quindi:
>      $$\Pr(B_2) \le 2e^{- \log(4Nk)} = \frac{2}{4Nk} = \frac{1}{2Nk} \le \frac{1}{2N}$$
>       dove l'ultima disuguaglianza è vera per $k \ge 1$.
>
> In definitiva, sfruttando Union Bound:
> $$\Pr(A) \leq \Pr(B_1 \cup B_2) \leq \Pr(B_1) + \Pr(B_2) \leq \frac{1}{2N} + \frac{1}{2N} = \frac{1}{N}$$
> quindi abbiamo dimostrato che con probabilità almeno $1 - \frac{1}{N}$, lo Spread di ogni item $i$ è al più $8t \log(4Nk) = O(t \log(Nk))$, ossia che il teorema è vero con alta probabilità.
>
> $\blacksquare$

#### Bound sul Load
Si ricorda che il load di un server $b$ è: 
$$Load_f(\mathcal{V}, b) = |\{i \in I : \exists V \in \mathcal{V} \text{ tale che } f_V(i) = b\}|$$
ossia il numero di item distinti che possono essere assegnati al server $b$ considerando tutte le view $V \in \mathcal{V}$ (**NB** si parla di server in $B$, quindi non delle copie: il Load è esercitato proprio sui server in sé)

Si ricorda che $n = |I|$ è il numero di item, $T = |\bigcup_{j=1}^k V_j|$ è il numero totale di server distinti presenti in tutte le view, $t$ è il parametro di DVH, $N$ è il parametro di confidenza, $m$ è il numero di copie virtuali per server, e $k$ è il numero di view.

> **Teorema 3 (Load)**
> 
> Sempre sotto DVH, per ogni server $b \in B$ si ha che $Load_{f \in \mathcal{UC}_{rnd}}(\mathcal{V}, b) = O(\frac{nt}{T} \log(Nmk))$ con probabilità almeno $1 - \frac{1}{N}$ rispetto alla scelta random di $f \in_u \mathcal{UC}_{rnd}$.
>
>Stiamo quindi affermando che usando consistent hashing con la famiglia $\mathcal{UC}_{rnd}$, nessun server riceve un carico eccessivo di item, anche considerando tutte le view.
>
> Analizziamo meglio questa formula: $O(\frac{nt}{T} \log(Nmk))$. Per DVH ogni view ha almeno $\frac{T}{t}$ server a disposizione. Il carico migliore che posso aspettarmi per questi server (distinti, qui non stiamo considerando le copie) è che nel distribuire gli $|I| = n$ item questi si vengano assegnati uniformemente tra i $\frac{T}{t}$ server, quindi che ogni server riceva in media $\frac{nt}{T}$ item --> la prima parte del bound è proprio questo valore, che rappresenta il carico ideale che ci aspetteremmo se gli item si distribuissero in modo perfettamente uniforme tra i server e che non possiamo battere. 
>
> La parte nel $\log$ ci descrive l'impatto di dover prendere in considerazione tante view: in particolare si cresce logaritmicamente rispetto al numero di view $k$ (ottimo, impatto solo logaritmico, si tollera bene la presenza di più view), numero di copie dei server $m$ (intuitivamente perché aumentare il numero di copie virtuali significa dover controllare per più server che il load sia ben distribuito, per questo si peggiora, ma solo logaritmicamente) e il parametro di confidenza $N$ (aumentando $N$ aumenta la probabilità che il teorema sia vero, ma peggiora anche il bound, però con crescita logaritmica).
>
> **Dimostrazione**
>
> Si dimostra il Teorema fissando un generico server $b \in B$ e stimando il numero di item che possono essere assegnati a quel server considerando tutte le view.  
> Fissiamo un server $b \in B$, questo server ha $m$ copie nel cerchio. Indichiamo con $p_b$ una generica copia di $b$. 
>
> Un item può essere assegnato a quella copia di $b$ solo se **il punto dell'item $r_I(i)$ cade nell'arco immediatamente a sinistra di $p_b$** in senso antiorario, senza che vi siano altri server point in mezzo (a cui altrimenti l'item sarebbe assegnato). Per stimare quindi quanti item possono finire su $b$ (e quindi il suo Load), dobbiamo stimare quanto sono lunghi questi archi di "responsabilità" delle sue $m$ copie.
>
> <img src="img/load.jpeg" alt="load" width="400"/>
>
> Fissiamo una copia $p_b$ di $b$. Consideriamo $S_b$ come l'arco che parte da $p_b$, procede in senso antiorario e rappresenta la "responsabilità" di $b$ (ossia, se un item cade in $S_b$ allora sarà assegnato a $p_b$ e quindi influirà sul suo load), vogliamo stimare la lunghezza di questo arco. 
>
> Per farlo sfruttiamo il Lemma 2 (Hitting Property): sappiamo infatti che probabilità almeno $1 - \frac{1}{2N}$, ogni view ha almeno un server point in un arco $S$ di lunghezza $s = \frac{4t\log(4Nk)}{Tm}$ --> se prendo $S_b$ di questa lunghezza avrò confidenza del fatto che con alta probabilità incontrerò un altro server point entro la fine dell'arco, e quindi ho una buona stima del mio arco di responsabilità.
>
> In particolare usiamo il Lemma con parametro di confidenza $N' = Nm$ (è per via del fatto che dobbiamo considerare tutte le $m$ copie di $b$, sarà chiara la scelta nel momento della semplificazione con union bound).  
> Applicando il Lemma con parametro $N'$, otteniamo che per una copia fissata $p_b$ l'arco necessario ha lunghezza al massimo:
> $$ s = \frac{4t\log(4Nmk)}{Tm} \qquad \text{w.p. } \quad 1 - \frac{1}{2N'} = 1- \frac{1}{2Nm} $$
> Vogliamo che la proprietà valga per tutte le $m$ copie di $b$ --> sfruttiamo Union Bound:
> $$\Pr(\text{esiste una copia } p_b \text{ tale che } S_{p_b} \text{ ha lunghezza maggiore di } s) \leq m \cdot \frac{1}{2Nm} = \frac{1}{2N}$$
> quindi definendo l'evento buono 
> $$A = \{\text{tutti gli archi di responsabilità delle } m \text{ copie di } b \text{ hanno lunghezza al più } s\}$$
> abbiamo che:
> $$\Pr(A) \ge 1 - \frac{1}{2N}$$
> $$\Pr(A^c) \le \frac{1}{2N}$$
> (NB. qui con "ha lunghezza maggiore di $s$ stiamo intendendo la probabilità che l'arco di responsabilità sia maggiore di $s$, e questo per la Hitting property come visto avviene con bassa probabilità. Ricorda che la lunghezza degli archi è deterministica, il punto è stimare con quale probabilità questi permettono di coprire un certo numero di server)
>
> A questo punto vogliamo stimare $\Pr(Load_f(\mathcal{V}, b) \ge z)$. Usando la legge delle probabilità totali, possiamo scrivere:
> $$\Pr(Load_f(\mathcal{V}, b) \ge z) = \Pr(Load_f(\mathcal{V}, b) \ge z | A) \cdot \Pr(A) + \Pr(Load_f(\mathcal{V}, b) \ge z | A^c) \cdot \Pr(A^c)$$
> Facendo un bound rozzo, stimiamo che $\Pr(Load_f(\mathcal{V}, b) \ge z | A^c) \le 1$ e $\Pr(A) \le 1$ (dato che ogni probabilità è al più 1). Invece sappiamo che scegliendo $s$ come visto $\Pr(A^C) \le \frac{1}{2N}$, quindi:
> $$\Pr(Load_f(\mathcal{V}, b) \ge z) \le \Pr(Load_f(\mathcal{V}, b) \ge z | A) + \frac{1}{2N}$$
>
> Dobbiamo stimare $\Pr(Load_f(\mathcal{V}, b) \ge z | A)$. Supponiamo che $A$ sia vero, allora ogni arco di responsabilità associato a una copia di $b$ ha lunghezza al più:
> $$s = \frac{4t\log(4Nmk)}{Tm}$$
> Nel caso peggiore, tutti questi archi di responsabilità sono disgiunti, quindi la regione del cerchio totale $R$ da cui $b$ può ricevere item misura al massimo:
> $$|R| \le ms = m \cdot \frac{4t\log(4Nmk)}{Tm} = \frac{4t\log(4Nmk)}{T}$$
> Definiamo ora la variabile aleatoria:
> $$X(b) = \text{numero di item i cui punti cadono in } R$$
> Poiché gli item sono mappati uniformemente e indipendentemente sul cerchio, $X(b) \sim Bin(n, |R|)$, con media $\mathbb{E}[X(b)] = n|R| \le \frac{4nt\log(4Nmk)}{T}$.
>
> Ma se A vale (si ricorda che A è l'evento per cui ogni arco di responsabilità ha lunghezza al più $s$), allora ogni item assegnato a $b$ deve necessariamente cadere in $R$, quindi:
> $$Load_f(\mathcal{V}, b) \le X(b)$$
> perciò abbiamo che:
> $$\Pr(Load_f(\mathcal{V}, b) \ge z | A) \le \Pr(X(b) \ge z)$$
> Si noti che abbiamo già mostrato sopra che il valore atteso di $X(b)$ è dell'ordine di $O(\frac{nt}{T} \log(Nmk))$, quindi abbiamo dimostrato che il load medio di $b$ è dell'ordine enunciato dal teorema. Vogliamo però un'analisi più forte, in concentrazione.
>
> Vogliamo mostrare che $X(b)$ non supera troppo la media, per farlo usiamo Chernoff upper tail (possiamo applicarlo perché $X(b)$ è variabile binomiale, quindi somma di Bernoulli indipendenti).
> $$\Pr(X(b) \ge (2 \mu)) \le e^{-\frac{\mu}{3}}$$
> Qui non si eseguono precisamente i calcoli, ma l'idea è sempre la stessa: si sceglie la soglia z come $O(\frac{nt}{T} \log(Nmk))$, quindi dell'ordine della media, e si mostra che la probabilità che $X(b)$ superi quella soglia decade esponenzialmente, quindi che X(b) ha alta probabilità di essere al più $O(\frac{nt}{T} \log(Nmk))$. Ciò implica, nel nostro caso, che anche il load di $b$ è al più $O(\frac{nt}{T} \log(Nmk))$ con alta probabilità.  
> Si arriverà quindi ad ottenere che, prendendo $c$ costante maggiore di 0:
> $$\Pr(X(b) \ge c \cdot \frac{nt}{T} \log(Nmk)) \le \frac{1}{2N}$$
> e quindi che:
> $$\begin{aligned}
> \Pr(Load_f(\mathcal{V}, b) \ge c \cdot \frac{nt}{T} \log(Nmk)) &\le \Pr(X(b) \ge c \cdot \frac{nt}{T} \log(Nmk)) + \frac{1}{2N} \\
> &\le \frac{1}{2N}+\frac{1}{2N} \le O\left(\frac{1}{N}\right)
> \end{aligned}$$ 
>
> $\blacksquare$

#### Bound sulla Balance
Si ricorda che la proprietà di **balance** dice intuitivamente che gli item devono distribuirsi in modo uniforme tra i server, di modo che non si creino server sovraccarichi.

Formalmente, per una view fissata $V$, una famiglia hash è balanced se:
$$Pr(f_V(i) = b) = \widetilde O\left(\frac{1}{|V|}\right) \qquad \forall i \in I, b \in V$$

Nel consistent hashing i server sono punti sul cerchio e ogni server è responsabile di un insieme di archi (per via delle copie), e gli item vengono assegnati al primo server che incontrano andando in senso orario. Ma quindi la probabilità che **un item finisca su un server $b$ dipende da quanto è grande la regione del cerchio controllata da $b$**. 

Per questo motivo si introduce:
$$M(b) = \text{misura totale della regione del cerchio di cui } b \text{ è responsabile}$$
Se $M(b)$ è grande allora $b$ sarà responsabile di molti item, al contrario se piccolo.

Si enunciano i Lemmi ma si omettono le dimostrazioni.

> **Lemma 3 $\left(M(b)\right)$**
>
> Sia $V \in \mathcal{V}$ una view. Allora, per ogni server $b \in V$ si ha che $M(b) = O(\frac{1}{|V|}(1+\frac{\log(N|V|)}{m}))$ con probabilità almeno $1 - \frac{1}{N}$.
>
> Interpretazione: quello che idealmente vorremmo è $M(b) \approx \frac{1}{|V|}$, ossia che ogni server sia responsabile di una regione uniforme del cerchio rispetto agli altri. Il Lemma ci dice che ci avviciniamo a questo se non per una piccola correzione logaritmica. Si noti che la crescita è logaritmica all'aumentare del parametro di confidenza e del numero di server, però si riduce all'aumentare del numero di copie $m$! Questo è perché intuitivamente più i punti si spargono sul cerchio, più il carico si uniforma (più copie --> balance migliore).
>
> **Lemma 4**
>
> Fissiamo una qualsiasi view $V \in \mathcal{V}$ e un qualsiasi item $i \in I$. Allora, per ogni server $b \in V$, la probabilità che $i$ sia assegnato a $b$ è:
> $$Pr(i \text{ è assegnato a } b) = O\left(\frac{1}{|V|} \left(1 + \frac{\log(N|V|)}{m}\right)\right) + \frac{1}{N}$$
> Questo deriva direttamente dal fatto che un item $i$ cade uniformemente sul cerchio, quindi la probabilità di finire su $b$ è proporzionale a $M(b)$.
>
> **Lemma 5**
>
> Infine, sotto la DVH, la probabilità che un qualsiasi item $i$ sia assegnato a un server $b$ in almeno una view è:
> $$Pr(i \text{ è assegnato a } b \text{ in almeno una view}) = O\left(\frac{t\log(Nk)}{T}\right) + \frac{1}{N}$$
> dove l'idea è che anche considerando più view, la probabilità che un item finisca in un server fissato resta piccola e cresce logaritmicamente con il numero di view $k$